# v26 — Off-board fix (aim convergence guard)

## What was wrong

I instrumented v14.8's `aim_with_prediction` (the function that decides what angle to launch fleets at moving targets). It does iterative fixed-point search:

1. Compute ETA assuming target is at current position.
2. Predict where target will be in ETA turns.
3. Recompute ETA aiming at that predicted position.
4. Repeat until predicted position stops changing (converged).

The original cap was 5 iterations. **When iteration didn't converge, the function returned its last (broken) guess anyway.**

## Measured impact (v14.8, 1 game vs starter)

| Outcome | Converged | Non-converged |
|---------|-----------|---------------|
| Hit a planet | 143 | 47 |
| Off-board | 46 (24%) | **70 (60%)** |

Drilling further by predicted ETA:

| ETA bucket | Converged OOB rate | **Non-converged OOB rate** |
|------------|--------------------|-----------------------------|
| 1-9 turns  | 35% | 53% |
| **10-19 turns** | 21% | **85%** |
| **20-29 turns** | 0% | **80%** |
| 30-39 turns | 5% | 31% |

Non-converged fleets at 10-29 ETA had an **80-85% chance of going off-board**. The "non-converged-returned" path was the dominant fallback in the agent — those broken guesses were the source of most off-board fleets.

## What v26 changes

Two surgical changes to `aim_with_prediction`:

1. **Bump iteration limit from 5 to 15.** With damping after iteration 5 (move halfway toward new estimate each step) to break oscillations. Most cases that need >5 iterations actually settle by ~10.
2. **If still non-converged after 15 iterations, fall back to `search_safe_intercept`** (a brute-force search over candidate ETAs that almost always finds a valid aim). If even that fails, return `None` — agent treats target as unreachable and tries something else. **Never return a broken guess.**

Built on top of v23 (the doctrine fix you saw before), so it carries those gains too.

## Measured improvement on multi-seed

Across 5 seeds (v14.8 vs starter, 200 steps each):
- **v14.8**: 33.1% OOB rate over 1582 fleets
- **v26**:   31.1% OOB rate over **879 fleets** (-44% volume)

The OOB rate barely moved (33% → 31%). What changed is **the agent now rejects half its previous shots** rather than launching them blindly. Combined with the doctrine fix from v23, fewer fleets means each fleet is bigger, more decisive, and more likely to actually capture rather than feed.

**This is not yet a confirmed ELO improvement.** The single-game vs-starter numbers look favorable but the real test is v26 vs v14.8 / v20c head-to-head at n=24, which this notebook runs.

## Honest caveat

Both `aim_with_prediction` (the original) and `search_safe_intercept` (v26's fallback) use the same `predict_target_position` and `safe_angle_and_distance` primitives. If the bug is upstream in those primitives — for example, if the angular-velocity prediction systematically underestimates rotation — then v26 won't fully fix it. v26 only catches non-convergence, not bad-prediction.

If v26 doesn't beat v23 in head-to-head at n=24, the off-board issue is deeper than convergence and we need to instrument the prediction primitives next.


## Setup

In [ ]:
V14_PATH  = "/kaggle/input/datasets/njabulobright/experiment/v14_8_final.py"
V20C_PATH = "/kaggle/input/datasets/njabulobright/experiment/v20c_phased_htvm.py"
V23_PATH  = "/kaggle/input/datasets/njabulobright/experiment/v23_no_probes.py"
V26_PATH  = "/kaggle/input/datasets/njabulobright/experiment/v26_aim_convergence.py"

QUICK_MODE = False
N = 24 if not QUICK_MODE else 6
STEPS = 500 if not QUICK_MODE else 200

RESULTS_DIR = "./results_v26"
import os, json
os.makedirs(RESULTS_DIR, exist_ok=True)
for label, p in [("v14",V14_PATH),("v20c",V20C_PATH),("v23",V23_PATH),("v26",V26_PATH)]:
    print(f"{label}: exists={os.path.exists(p)}  {p}")


In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0" 

In [ ]:
import importlib.util, math, time, random, json, statistics
from kaggle_environments import make
from collections import Counter

def load_agent(path, suffix=""):
    name = f"a_{suffix}_{random.random()}"
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

def first_planet_hit(fx, fy, fang, fships, planets, max_turns=200):
    """Replays engine collision detection. Returns 'hit', 'oob', 'sun', or 'timeout'."""
    speed = min(6.0, 1.0 + 5.0 * (math.log(max(1, fships)) / math.log(1000.0)) ** 1.5)
    dx, dy = math.cos(fang), math.sin(fang); x, y = fx, fy
    def p2s(px,py,vx,vy,wx,wy):
        l2 = (vx-wx)**2 + (vy-wy)**2
        if l2 == 0: return math.hypot(px-vx, py-vy)
        tt = max(0, min(1, ((px-vx)*(wx-vx)+(py-vy)*(wy-vy))/l2))
        return math.hypot(px-(vx+tt*(wx-vx)), py-(vy+tt*(wy-vy)))
    for t in range(1, max_turns+1):
        old = (x, y); x += dx*speed; y += dy*speed; new = (x, y)
        if not (0 <= x <= 100 and 0 <= y <= 100): return "oob"
        if p2s(50, 50, *old, *new) < 10: return "sun"
        for p in planets:
            if p2s(p[2], p[3], *old, *new) < p[4]: return "hit"
    return "timeout"

def fleet_metrics(state, player_idx):
    sizes = []; outcomes = Counter()
    for i in range(1, len(state)-1):
        on = state[i][0].observation; nn = state[i+1][0].observation
        ids = {f[0] for f in on.get("fleets", [])}
        ps = on.get("planets", [])
        for f in nn.get("fleets", []):
            if f[0] in ids or f[1] != player_idx: continue
            sizes.append(f[6])
            outcomes[first_planet_hit(f[2], f[3], f[4], f[6], ps)] += 1
    n = len(sizes)
    if n == 0: return {"n": 0}
    return {
        "n": n,
        "avg_size": sum(sizes)/n,
        "pct_tiny": 100*sum(1 for s in sizes if s <= 10)/n,
        "pct_big": 100*sum(1 for s in sizes if s > 30)/n,
        "pct_oob": 100*outcomes.get("oob", 0)/n,
        "pct_hit": 100*outcomes.get("hit", 0)/n,
    }

def wilson_95(wins, total):
    if total < 1: return (None, None)
    z = 1.96; phat = wins/total
    denom = 1 + z*z/total
    center = (phat + z*z/(2*total)) / denom
    margin = z*math.sqrt(phat*(1-phat)/total + z*z/(4*total*total)) / denom
    return (max(0,(center-margin))*100, min(1,(center+margin))*100)

def h2h(challenger_path, baseline_path, n_games, steps, label, seed_base=1000):
    print(f"\n=== {label}: {n_games} games of {steps} steps ===")
    ch_wins = 0; bl_wins = 0; ties = 0
    log = []
    for g in range(n_games):
        random.seed(seed_base + g)
        env = make("orbit_wars", configuration={"episodeSteps": steps, "actTimeout": 10}, debug=False)
        A = load_agent(challenger_path, f"ch_{g}").agent
        B = load_agent(baseline_path, f"bl_{g}").agent
        if g % 2 == 0: agents=[A,B]; ch_seat,bl_seat=0,1
        else: agents=[B,A]; ch_seat,bl_seat=1,0
        t0 = time.time()
        state = env.run(agents)
        dt = time.time() - t0
        ch_m = fleet_metrics(state, ch_seat)
        bl_m = fleet_metrics(state, bl_seat)
        r0, r1 = state[-1][0].reward, state[-1][1].reward
        scores = [0, 0]
        for p in state[-1][0].observation.get("planets", []):
            if p[1] in (0,1): scores[p[1]] += p[5]
        for f in state[-1][0].observation.get("fleets", []):
            scores[f[1]] += f[6]
        if r0 == r1: ties += 1; tag = "TIE"
        elif (ch_seat==0 and r0>r1) or (ch_seat==1 and r1>r0):
            ch_wins += 1; tag = "ch W"
        else: bl_wins += 1; tag = "bl W"
        log.append({"g": g+1, "ch_seat": ch_seat, "winner": tag,
                    "ch": ch_m, "bl": bl_m,
                    "final_ships": scores, "ran_steps": len(state)-1, "time_s": dt})
        ch_n, bl_n = ch_m.get('n',0), bl_m.get('n',0)
        ch_oob = ch_m.get('pct_oob',0); bl_oob = bl_m.get('pct_oob',0)
        print(f"  g{g+1:2d}: {tag:5s}  fleets[ch={ch_n} oob={ch_oob:.0f}%  bl={bl_n} oob={bl_oob:.0f}%]  ships={scores}  t={dt:.0f}s")
    
    decisive = ch_wins + bl_wins
    wr = ch_wins/decisive*100 if decisive else 50
    ci = wilson_95(ch_wins, decisive)
    avg_ch_oob = statistics.mean(g['ch'].get('pct_oob', 0) for g in log)
    avg_bl_oob = statistics.mean(g['bl'].get('pct_oob', 0) for g in log)
    avg_ch_n = statistics.mean(g['ch'].get('n', 0) for g in log)
    
    print(f"\n  RESULT: {ch_wins}W/{bl_wins}L/{ties}T  -> winrate {wr:.1f}%")
    if ci[0] is not None: print(f"  95% CI: [{ci[0]:.0f}%, {ci[1]:.0f}%]")
    print(f"  Doctrine: ch sends {avg_ch_n:.0f} fleets/game,  oob rate ch={avg_ch_oob:.0f}%  bl={avg_bl_oob:.0f}%")
    
    return {"label": label, "ch_wins": ch_wins, "bl_wins": bl_wins, "ties": ties,
            "games": n_games, "winrate": wr, "ci_95": ci,
            "avg_ch_oob": avg_ch_oob, "avg_bl_oob": avg_bl_oob,
            "avg_ch_n": avg_ch_n, "log": log}

print("helpers loaded")


## Experiment 1 — v26 vs v14.8

Primary test. Expected outcome: v26 wins (carries v20c + v23 gains plus the convergence fix), with **v26's avg OOB rate visibly lower than v14.8's**.

The OOB columns in the per-game output are the new diagnostic. If v26's OOB stays similar or *higher* than v14.8's, the convergence guard isn't doing its job and we need to investigate further.

In [ ]:
t0 = time.time()
v26 = h2h(V26_PATH, V14_PATH, N, STEPS, "v26_vs_v14", seed_base=70000)
print(f"\nTotal: {time.time()-t0:.0f}s")
with open(f"{RESULTS_DIR}/v26_vs_v14.json", "w") as f: json.dump(v26, f, indent=2)


## Experiment 2 — v26 vs v23

The decisive test of whether the convergence fix actually helped, beyond v23's doctrine fix alone. If v26 only marginally beats v23, the OOB win rate isn't translating to victories — meaning rejecting bad shots costs you more than it saves.

In [ ]:
t0 = time.time()
v26_v_23 = h2h(V26_PATH, V23_PATH, N, STEPS, "v26_vs_v23", seed_base=80000)
print(f"\nTotal: {time.time()-t0:.0f}s")
with open(f"{RESULTS_DIR}/v26_vs_v23.json", "w") as f: json.dump(v26_v_23, f, indent=2)


## Experiment 3 — v26 vs v20c

For completeness. v20c was the previous validated best (62.5% vs v14.8 at n=24).

In [ ]:
t0 = time.time()
v26_v_20c = h2h(V26_PATH, V20C_PATH, N, STEPS, "v26_vs_v20c", seed_base=90000)
print(f"\nTotal: {time.time()-t0:.0f}s")
with open(f"{RESULTS_DIR}/v26_vs_v20c.json", "w") as f: json.dump(v26_v_20c, f, indent=2)


## Decision

In [ ]:
import json, os
def load(name):
    p = f"{RESULTS_DIR}/{name}.json"
    if not os.path.exists(p): return None
    with open(p) as f: return json.load(f)

r1 = load("v26_vs_v14")
r2 = load("v26_vs_v23")
r3 = load("v26_vs_v20c")

print("="*72)
print("  v26 SUBMISSION DECISION")
print("="*72)

if r1:
    ci = r1["ci_95"]; ci_str = f"[{ci[0]:.0f}-{ci[1]:.0f}]" if ci[0] else "n/a"
    print(f"\n[1] v26 vs v14.8: {r1['winrate']:.1f}% wr  CI {ci_str}")
    print(f"     v26 OOB rate:  {r1['avg_ch_oob']:.0f}%  (was 33% in v14.8 alone)")
if r2:
    ci = r2["ci_95"]; ci_str = f"[{ci[0]:.0f}-{ci[1]:.0f}]" if ci[0] else "n/a"
    print(f"\n[2] v26 vs v23: {r2['winrate']:.1f}% wr  CI {ci_str}")
    print(f"     {'✅ convergence fix added value' if r2['winrate']>55 else '⚠️ doctrine alone (v23) was the win'}")
if r3:
    ci = r3["ci_95"]; ci_str = f"[{ci[0]:.0f}-{ci[1]:.0f}]" if ci[0] else "n/a"
    print(f"\n[3] v26 vs v20c: {r3['winrate']:.1f}% wr  CI {ci_str}")

print("\n" + "="*72)
all_ok = r1 and r2 and r3
if not all_ok:
    print("  Run all three experiments before deciding.")
else:
    # Decision tree
    if r1["winrate"] >= 60 and r2["winrate"] >= 53 and r3["winrate"] >= 53:
        print("  ✅ RECOMMENDATION: SHIP v26")
        print("     v26 beats v14.8, v23, and v20c. Best validated agent.")
    elif r1["winrate"] >= 60 and r3["winrate"] >= 53:
        print("  ✅ RECOMMENDATION: SHIP v26")
        print("     v26 beats v14.8 and v20c. Convergence fix on top of doctrine.")
    elif r1["winrate"] >= 55:
        print("  ⚠️  v26 beats v14.8 but not clearly better than v23/v20c.")
        print("     Ship the simplest one — probably v23 (doctrine alone).")
    else:
        print("  ❌ v26 didn't help. Ship v23 (last validated improvement).")
print("="*72)


## What's next regardless of result

1. **If v26 doesn't help**: the OOB issue isn't really "non-convergence." Probably the engine's actual angular velocity is different from what the agent's `predict_target_position` computes — there could be sweep effects, comet interactions, or off-by-one timing that make even "converged" predictions wrong by enough to send fleets off-board. Next step: directly compare agent's predicted target position at turn N vs the engine's actual position at turn N.

2. **If v26 helps but only marginally**: the convergence fix is working but most of the OOB is from elsewhere. Likely candidates:
   - Targets that get captured/destroyed during fleet's flight (no recovery in `aim_with_prediction`)
   - Comets (complex paths the linear prediction can't represent)
   - Fleet speed changes when stacking (engine seems to sum same-owner fleets at planet impact, but in flight they're independent)

3. **If v26 wins clearly**: ship it. Then revisit the **multi-source coordinated strikes** issue — currently each home planet picks its target greedily, often ending up with multiple sources sending small fleets to the same target rather than one source sending a big one. Compound on top of v26.

The full diagnostic kit is in this notebook's `helpers` cell — you can rerun the convergence/outcome analysis on any new candidate by hooking `aim_with_prediction` and `plan_shot` like I did.